In [0]:
password = dbutils.secrets.get(scope = "scope", key = "scopekey")
print(password)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS global_wholesale_distributor;
USE CATALOG global_wholesale_distributor;
 
CREATE SCHEMA IF NOT EXISTS raw_wholesale_distributor;
CREATE SCHEMA IF NOT EXISTS staging_wholesale_distributor;
CREATE SCHEMA IF NOT EXISTS curated_wholesale_distributor;

In [0]:
storage_account_name = "deprojectaccount"
container_name = "decontainer"
storage_access_key = dbutils.secrets.get(scope = "scope", key = "scopekey")
storage_config_key = f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net"

In [0]:
from pyspark.sql.functions import current_timestamp, col
 
def ingest_to_bronze(file_name, table_name):
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option(storage_config_key, storage_access_key) \
        .load(f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{file_name}")
   
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"global_wholesale_distributor.raw_wholesale_distributor.{table_name}")

In [0]:
ingest_to_bronze("customers.csv", "customers")
ingest_to_bronze("products.csv", "products")
ingest_to_bronze("invoices.csv", "invoices")
ingest_to_bronze("invoice_line_items.csv", "invoice_line_items")
ingest_to_bronze("payments.csv", "payments")
ingest_to_bronze("exchange_rates.csv", "exchange_rates")
ingest_to_bronze("regions.csv", "regions")